# Word2Vec 단어 임베딩과 FastText 비교

Word2Vec 으로 한국어 미니 코퍼스를 학습해 단어를 밀집 벡터로 표현하고, `most_similar` / `similarity` 로 단어 간 의미 관계를 살펴봅니다. 이어서 FastText 를 학습해 OOV(미등록 단어) 처리에서의 차이를 비교합니다.

## 학습 목표
- Word2Vec(CBOW / Skip-gram) 의 기본 아이디어를 이해한다
- gensim 의 `Word2Vec` API 로 임베딩을 학습/조회한다
- 단어 벡터 간 코사인 유사도 의미를 해석한다
- Word2Vec 의 OOV 한계와 FastText 의 서브워드 해결책을 비교한다

## 사용 라이브러리
- `gensim` (`Word2Vec`, `FastText`)

In [1]:
!pip install gensim

## 1. 학습용 코퍼스 준비

토큰화된 문장 리스트를 준비합니다. 강아지(쵸키)와 푸들에 관한 짧은 한국어 문장 7개로 구성된 미니 코퍼스로, 단어 임베딩의 기본 동작을 빠르게 확인하기 위한 용도입니다.

In [2]:
import gensim
from gensim.models.word2vec import Word2Vec

sentences = [
                ['이', '강아지는', '털',  '색깔이', '갈색이다'],
                ['나는',   '갈색', '강아지를',   '좋아한다'],
                ['푸들은',   '갈색', '털이',   '많다'],
                ['갈색',   '털을', '가진',  '푸들', '강아지'],
                ['푸들은',   '털이', '빠지지',   '않아', '자주', '미용을', '해야',' 한다'],
                ['갈색', '털이',  '있는', '강아지가', '꼬리를', '친다'],
                ['쵸키는', '내가', '키우는', '강아지다']
            ]

print(sentences)

[['이', '강아지는', '털', '색깔이', '갈색이다'], ['나는', '갈색', '강아지를', '좋아한다'], ['푸들은', '갈색', '털이', '많다'], ['갈색', '털을', '가진', '푸들', '강아지'], ['푸들은', '털이', '빠지지', '않아', '자주', '미용을', '해야', ' 한다'], ['갈색', '털이', '있는', '강아지가', '꼬리를', '친다'], ['쵸키는', '내가', '키우는', '강아지다']]


In [3]:
# 주어진 sentences를 가지고 학습하기
model = Word2Vec(sentences=sentences, min_count = 1, vector_size=10, window=2)
print(model)

# 토큰의 벡터 크기
model.wv.vectors.shape

Word2Vec<vocab=30, vector_size=10, alpha=0.025>


(30, 10)

## 2. Word2Vec 모델 학습

`gensim.models.Word2Vec` 으로 토큰 리스트를 학습합니다. CBOW 와 Skip-gram 두 가지 방식이 있으며 기본은 CBOW(`sg=0`) 입니다.

| 파라미터 | 의미 |
|---------|------|
| `sentences` | 토큰화된 문장 리스트 |
| `vector_size` | 임베딩 벡터 차원 |
| `window` | 좌우 컨텍스트 크기 |
| `min_count` | 최소 등장 빈도 (이하 무시) |
| `sg` | 0=CBOW, 1=Skip-gram |

| 학습 방식 | 입력 | 출력 |
|----------|------|------|
| CBOW | 주변 단어들 | 중심 단어 |
| Skip-gram | 중심 단어 | 주변 단어들 |

In [4]:
# vocab 알아보기
vocab = model.wv.index_to_key
print(vocab)
print(len(vocab))

['갈색', '털이', '푸들은', '강아지다', '키우는', '내가', '쵸키는', '친다', '꼬리를', '강아지가', '있는', ' 한다', '해야', '미용을', '자주', '않아', '빠지지', '강아지', '푸들', '가진', '털을', '많다', '좋아한다', '강아지를', '나는', '갈색이다', '색깔이', '털', '강아지는', '이']
30


In [19]:
# 유사도 알아보기
model.wv.most_similar('푸들')

[('강아지다', 0.7671629190444946),
 ('이', 0.5327391028404236),
 ('빠지지', 0.4261683225631714),
 ('내가', 0.30405837297439575),
 ('친다', 0.24754926562309265),
 ('있는', 0.21394889056682587),
 ('강아지', 0.1807485818862915),
 ('강아지가', 0.14403586089611053),
 ('나는', 0.12485264986753464),
 ('갈색이다', 0.11776044964790344)]

In [5]:
model.wv.most_similar('푸들', topn=5)


[('강아지다', 0.7671629190444946),
 ('이', 0.5327391028404236),
 ('빠지지', 0.4261683225631714),
 ('내가', 0.30405837297439575),
 ('친다', 0.24754926562309265)]

## 3. OOV 와 단어 간 유사도

`most_similar` 로 학습된 단어와 가장 가까운 단어를 찾을 수 있습니다. 그러나 학습 코퍼스에 등장하지 않은 단어(예: '프돌')는 `KeyError` 가 발생합니다 — Word2Vec 의 대표적인 한계인 OOV(Out-Of-Vocabulary) 문제입니다. `similarity` 로는 두 단어 벡터의 코사인 유사도를 직접 계산할 수 있습니다.

In [6]:
model.wv.most_similar('프돌', topn=5)

KeyError: "Key '프돌' not present in vocabulary"

In [8]:
model.wv.similarity('갈색', '푸들')

np.float32(-0.3080608)

#FastText 모델

## 4. FastText 모델 학습

FastText 는 Word2Vec 을 확장해 **단어를 문자 n-gram 의 합** 으로 표현합니다. 동일한 코퍼스/하이퍼파라미터로 학습하되, OOV 단어 처리에서 차이를 보입니다.

| 모델 | 단위 | OOV |
|------|------|-----|
| Word2Vec | 단어 | KeyError |
| FastText | 단어 + char n-gram | 벡터 합성 가능 |

In [9]:
from gensim.models.fasttext import FastText

In [10]:
# 주어진 sentences를 가지고 학습하기
model = FastText(sentences=sentences, min_count = 1, vector_size=10, window=2)
print(model)

# 토큰의 벡터 크기
model.wv.vectors.shape

FastText<vocab=30, vector_size=10, alpha=0.025>


(30, 10)

In [11]:
# vocab 알아보기
vocab = model.wv.index_to_key
print(vocab)
print(len(vocab))

['갈색', '털이', '푸들은', '강아지다', '키우는', '내가', '쵸키는', '친다', '꼬리를', '강아지가', '있는', ' 한다', '해야', '미용을', '자주', '않아', '빠지지', '강아지', '푸들', '가진', '털을', '많다', '좋아한다', '강아지를', '나는', '갈색이다', '색깔이', '털', '강아지는', '이']
30


### FastText 의 OOV 강건성

같은 코퍼스로 학습했지만, '프돌' 처럼 vocab 에 없는 단어를 입력해도 FastText 는 문자 n-gram 으로 벡터를 합성해 유사 단어를 반환합니다. 이는 한국어 조사·활용형 같은 형태 변화에 특히 유용합니다.

In [12]:
model.wv.most_similar('프돌', topn=5)

[('강아지는', 0.634107768535614),
 ('친다', 0.36526361107826233),
 ('내가', 0.3577108681201935),
 ('미용을', 0.3072504699230194),
 ('강아지를', 0.302501380443573)]

In [13]:
model.wv.most_similar('푸들', topn=5)

[('강아지', 0.599902331829071),
 ('키우는', 0.5892254710197449),
 ('강아지가', 0.43106693029403687),
 ('강아지는', 0.40004003047943115),
 ('강아지다', 0.3329114317893982)]

In [15]:
vector = model.wv['푸들']
print(vector)
print(f"벡터 크기: {vector.shape}")

[-0.07626004 -0.04895312 -0.022177   -0.01393136 -0.0021206  -0.01079516
  0.00050327 -0.00523408  0.00737218 -0.0024851 ]
벡터 크기: (10,)


## 5. OOV(미등록 단어) 벡터 생성

FastText 는 학습 어휘에 없는 '푸들강아지' 같은 합성어도 문자 n-gram 의 합으로 벡터를 만들고 유사 단어를 검색할 수 있습니다. Word2Vec 이 KeyError 를 던지던 상황과의 차이를 직접 확인합니다.

In [ ]:
# OOV 단어의 벡터를 확인하기
vector = model.wv['푸들강아지']
print(vector)

[-0.01200884 -0.00629554 -0.01389806  0.0080246  -0.00957612  0.02290094
  0.0096917   0.00475003 -0.01175262  0.0175662 ]


In [18]:
# OOV 단어의 유사 단어 찾기
model.wv.most_similar('푸들강아지', topn=5)

[('않아', 0.7465896606445312),
 ('꼬리를', 0.5410068035125732),
 ('갈색', 0.45234668254852295),
 ('자주', 0.4128827154636383),
 ('털을', 0.3748236894607544)]